In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm

from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import r2_score, mean_squared_error

import utils
import paths as p

In [2]:
glm_dir = Path(p.DATA_DIR) / 'glm_encoding'

model_dir = glm_dir / 'glm_models'
model_dir.mkdir(parents=True, exist_ok=True)

plots_dir = glm_dir / 'glm_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

units_vetted = pd.read_csv(os.path.join(p.DATA_DIR, 'units_vetted.csv'), index_col=0).sort_values('unit_id')

In [3]:
events, trials, spikes, _ = utils.get_data_for_debugging(units_vetted)

### make spike time array

In [4]:
def generate_spike_counts(spikes, spike_bin_size):
    """
    Generate spike counts and bin centers from spike times.
    returns non-zero spike counts and their corresponding bin centers.
    """
    spike_times = spikes.spike_time.values
    bounds = (
        np.round(spike_times.min(), decimals=1),
        np.round(spike_times.max(), decimals=1)
    )
    spike_bin_edges = np.arange(
        bounds[0] - spike_bin_size,
        bounds[1] + 2*spike_bin_size,
        spike_bin_size
    )
    spike_bin_centers = spike_bin_edges[:-1] + spike_bin_size/2
    spike_counts = np.histogram(spike_times, bins=spike_bin_edges)[0]

    return spike_counts, spike_bin_centers

In [5]:
# spike_bin_size = 0.1  # seconds
# spike_counts, spike_bin_centers = generate_spike_counts(spikes, spike_bin_size)

### make design matrix

In [6]:
def modify_events(events, trials):
    # Map trial_id to rewarded status
    rewarded_map = trials.set_index('trial_id')['rewarded']

    events_mod = events.copy()
    # Find lick_cons events
    mask_lick_cons = events_mod['event_type'] == 'lick_cons'

    # Map rewarded status for lick_cons events
    rewarded_status = events_mod.loc[mask_lick_cons, 'trial_id'].map(rewarded_map)

    # Update event_type based on rewarded status
    events_mod.loc[mask_lick_cons & (rewarded_status == True), 'event_type'] = 'lick_cons_reward'
    events_mod.loc[mask_lick_cons & (rewarded_status == False), 'event_type'] = 'lick_cons_no_reward'
    return events_mod

In [7]:
def generate_event_bin_templates(sweep_window, event_bin_size):
    """ 
    Generate templates for event bin edges and centers.
    """
    event_bin_edges_template = np.arange(-sweep_window - event_bin_size/2, sweep_window + event_bin_size, event_bin_size)
    event_bin_centers_template = event_bin_edges_template[:-1] + event_bin_size/2
    return event_bin_edges_template, event_bin_centers_template

In [8]:
def generate_event_bins(spike_bin_centers, event_bin_edges_template, event_bin_centers_template):
    """
    Generate event bins and their centers based on template.
    """
    event_bin_edges = spike_bin_centers[:, None] + event_bin_edges_template[None, :]
    event_bin_centers = spike_bin_centers[:, None] + event_bin_centers_template[None, :]
    return event_bin_edges, event_bin_centers

In [9]:
def build_design_matrix(events, spike_bin_centers, event_bin_edges_template, event_bin_centers_template, encoding_events):
    """
    For each event type in encoding_events, create a design matrix (one-hot or count)
    and stack them horizontally into a single big design matrix.
    """
    event_bin_edges, event_bin_centers = generate_event_bins(spike_bin_centers, event_bin_edges_template, event_bin_centers_template)
    X_list = []
    for event_type in encoding_events:
        event_rows = events[events.event_type == event_type]
        X_event = np.zeros_like(event_bin_centers, dtype=int)
        for _, event in event_rows.iterrows():
            mask = (event_bin_edges[:, :-1] < event.event_end_time) & (event_bin_edges[:, 1:] > event.event_start_time)
            X_event += mask.astype(int)
        X_list.append(X_event)
    X_all = np.hstack(X_list) # Stack all event matrices side by side
    return X_all

In [10]:
# sweep_window = 10 # seconds
# event_bin_size = 0.1 # seconds
# encoding_events = ['visual', 'lick_bg', 'lick_wait', 'lick_cons_reward', 'lick_cons_no_reward']

# event_bin_edges_template, event_bin_centers_template = generate_event_bin_templates(sweep_window, event_bin_size)
# X_all = build_design_matrix(events, trials, spike_bin_centers, event_bin_edges_template, event_bin_centers_template, encoding_events)

### GLM

In [11]:
def fit_poisson_glm(unit_id,
                    spikes, events, 
                    spike_bin_size, 
                    event_bin_edges_template, event_bin_centers_template,
                    encoding_events,
                    alpha, max_iter,
                    model_dir):
    """
    Fit a Poisson GLM to the spike data using the provided events and trials.
    Returns a dictionary with the fitted model and performance metrics.
    """
    spike_counts, spike_bin_centers = generate_spike_counts(spikes, spike_bin_size)
    X_all = build_design_matrix(events, spike_bin_centers, event_bin_edges_template, event_bin_centers_template, encoding_events)

    X = X_all.astype(float)
    y = spike_counts.astype(float)

    glm = PoissonRegressor(alpha=alpha, max_iter=max_iter)
    glm.fit(X, y)

    joblib.dump(glm, model_dir / f"glm_{unit_id}.joblib")

    y_pred = glm.predict(X)
    glm_dict = {
        'unit_id': unit_id,
        'x': X_all,
        'y': y,
        'y_bin_centers': spike_bin_centers,
        'y_pred': y_pred,
        'alpha': alpha,
        'max_iter': max_iter,
        'r2': r2_score(y, y_pred),
        'mse': mean_squared_error(y, y_pred),
        'coefficients': glm.coef_,
        'intercept': glm.intercept_,
    }
    # Save glm_dict as pickle
    joblib.dump(glm_dict, model_dir / f"glm_results_{unit_id}.pkl")
    
    return glm_dict

In [12]:
def plot_glm_results(glm_dict, event_bin_centers_template, encoding_events):
    n_events = len(encoding_events)
    n_bins = glm_dict['x'].shape[1] // n_events

    fig_height = 2.5 * n_events + 3
    fig, axes = plt.subplots(n_events + 1, 1, figsize=(12, fig_height), sharex=False)

    # Top: GLM weights per event
    for i, event in enumerate(encoding_events):
        start = i * n_bins
        end = (i + 1) * n_bins
        axes[i].plot(event_bin_centers_template, glm_dict['coefficients'][start:end])
        axes[i].set_ylabel('Weight')
        axes[i].set_title(f'GLM Weights: {event}')
        axes[i].axhline(0, color='gray', linestyle='--', linewidth=0.5)
        axes[i].set_xlim(event_bin_centers_template.min(), event_bin_centers_template.max())

    # Bottom: Predicted vs actual spike count
    ax_pred = axes[-1]
    ax_pred.plot(glm_dict['y_bin_centers'], glm_dict['y'], label='Actual', color='black', alpha=0.6)
    ax_pred.plot(glm_dict['y_bin_centers'], glm_dict['y_pred'], label='Predicted', color='red', linestyle='--')
    ax_pred.set_xlabel('Time (s)')
    ax_pred.set_xlim(glm_dict['y_bin_centers'].min(), glm_dict['y_bin_centers'].max())

    # Annotate with metrics
    annotation = (f"α = {glm_dict['alpha']:.3f}  | max_iter = {glm_dict['max_iter']} | R² = {glm_dict['r2']:.3f}  |  MSE = {glm_dict['mse']:.3f}")
    ax_pred.text(0.01, 0.95, annotation, transform=ax_pred.transAxes,
                    fontsize=10, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

    plt.tight_layout()
    fig.savefig(plots_dir / f"{glm_dict['unit_id']}_glm_plot.png")
    plt.close(fig)

In [ ]:
# # parameters
# spike_bin_size = 0.1  # seconds
# sweep_window = 10 # seconds
# event_bin_size = 0.1 # seconds
# encoding_events = ['visual', 'lick_bg', 'lick_wait', 'lick_cons_reward', 'lick_cons_no_reward']

# alpha = 1.0
# max_iter = 1000

# event_bin_edges_template, event_bin_centers_template = generate_event_bin_templates(sweep_window, event_bin_size)

In [ ]:
# events = modify_events(events, trials)
# glm_dict = fit_poisson_glm("test",
#                            spikes, events,
#                            spike_bin_size, 
#                            event_bin_edges_template, event_bin_centers_template,
#                            encoding_events,
#                            alpha, max_iter,
#                            model_dir)

In [ ]:
# plot_glm_results(glm_dict, event_bin_centers_template, encoding_events)

GPT attempt to search best parameters

In [16]:
# # Make sure inputs are float for sklearn
# X = X_all.astype(float)
# y = spike_counts.astype(float)

# # Build pipeline: optional scaler (safe for Poisson), regressor
# pipeline = Pipeline([
#     ('scaler', StandardScaler()),  # Not strictly required for Poisson, but can help for interpretability
#     ('poisson', PoissonRegressor(max_iter=1000))
# ])

# # Grid for alpha (L2 regularization strength)
# param_grid = {
#     'poisson__alpha': np.logspace(-4, 2, 10)  # Search from very low to moderate regularization
# }

# # Poisson deviance is appropriate for Poisson-distributed targets
# scorer = make_scorer(mean_poisson_deviance, greater_is_better=False)

# # Cross-validation setup
# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# # Fit with grid search
# search = GridSearchCV(pipeline, param_grid, cv=cv, scoring=scorer, n_jobs=-1, verbose=1)
# search.fit(X, y)

# # Best model
# best_model = search.best_estimator_

# # Print results
# print(f"Best alpha: {search.best_params_['poisson__alpha']}")
# print(f"CV score (mean Poisson deviance): {-search.best_score_:.3f}")

# y_pred = best_model.predict(X)
# print(f"R² score: {r2_score(y, y_pred):.3f}")
# print(f"Mean squared error: {mean_squared_error(y, y_pred):.3f}")

### looping

In [17]:
# parameters
spike_bin_size = 0.1  # seconds
sweep_window = 10 # seconds
event_bin_size = 0.1 # seconds
encoding_events = ['visual', 'lick_bg', 'lick_wait', 'lick_cons_reward', 'lick_cons_no_reward']

alpha = 1.0
max_iter = 1000

event_bin_edges_template, event_bin_centers_template = generate_event_bin_templates(sweep_window, event_bin_size)

In [ ]:
# for debugging
units_short= units_vetted.head(2)
units_grouped = units_vetted.groupby("session_id")

: 

In [ ]:
failed_units = []
units_grouped = units_vetted.groupby('session_id')

for session_id, session_units in tqdm(units_grouped, desc="Sessions"):
    events, trials, units = utils.get_session_data(session_id)
    events = modify_events(events, trials)
    for _, unit_info in tqdm(session_units.iterrows(), total=len(session_units), desc=f"Units in {session_id}", leave=False):
        unit_id = unit_info['unit_id']
        spikes = units[unit_info['id']]
        try:
            unit_model_path = model_dir / f"glm_{unit_id}.joblib"
            if unit_model_path.exists():
                print(f"Model for {unit_id} already exists, skipping.")
                continue
            
            glm_dict = fit_poisson_glm(
                unit_id,
                spikes, events,
                spike_bin_size,
                event_bin_edges_template, event_bin_centers_template,
                encoding_events,
                alpha, max_iter,
                model_dir
            )
            
            plot_glm_results(glm_dict, event_bin_centers_template, encoding_events)
        except Exception as e:
            print(f"Failed to fit GLM for unit {unit_id} in session {session_id}: {e}")
            failed_units.append((unit_id, str(e)))

Sessions:   0%|          | 0/93 [00:00<?, ?it/s]

Model for RZ034_2024-07-13_str_unit-106 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-109 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-110 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-111 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-112 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-113 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-116 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-117 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-119 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-121 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-126 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-127 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-13 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-135 already exists, skipping.
Model for RZ034_2024-07-13_str_unit-14 already exists, skipping.
Model for RZ

Model for RZ034_2024-07-14_str_unit-105 already exists, skipping.
Model for RZ034_2024-07-14_str_unit-108 already exists, skipping.
Model for RZ034_2024-07-14_str_unit-126 already exists, skipping.
Model for RZ034_2024-07-14_str_unit-130 already exists, skipping.
Model for RZ034_2024-07-14_str_unit-131 already exists, skipping.


Sessions:  26%|██▌       | 24/93 [6:39:28<7:37:30, 397.83s/it]  

Failed to fit GLM for unit RZ050_2024-11-20_str_unit-335 in session RZ050_2024-11-20_str: [Errno 28] No space left on device


Failed to fit GLM for unit RZ050_2024-11-20_str_unit-336 in session RZ050_2024-11-20_str: [Errno 28] No space left on device


Failed to fit GLM for unit RZ050_2024-11-20_str_unit-337 in session RZ050_2024-11-20_str: [Errno 28] No space left on device
